In [1]:
import pandas as pd
import numpy as np

In [2]:
!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv

--2025-10-14 09:38:08--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8001::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.01s   

2025-10-14 09:38:08 (5.80 MB/s) - ‘course_lead_scoring.csv’ saved [80876/80876]



In [3]:
data = pd.read_csv("course_lead_scoring.csv")

In [4]:
data

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1
...,...,...,...,...,...,...,...,...,...
1457,referral,manufacturing,1,NaN,self_employed,north_america,4,0.53,1
1458,referral,technology,3,65259.0,student,europe,2,0.24,1
1459,paid_ads,technology,1,45688.0,student,north_america,3,0.02,1
1460,referral,NaN,5,71016.0,self_employed,north_america,0,0.25,1


In [7]:
data["industry"].mode()[0]

'retail'

In [8]:
numeric_df = data.select_dtypes(include=['number'])
correlation_matrix = numeric_df.corr()
print("Correlation Matrix:\n")
print(correlation_matrix)

Correlation Matrix:

                          number_of_courses_viewed  annual_income  \
number_of_courses_viewed                  1.000000       0.031551   
annual_income                             0.031551       1.000000   
interaction_count                        -0.023565       0.048618   
lead_score                               -0.004879       0.005334   
converted                                 0.435914       0.078256   

                          interaction_count  lead_score  converted  
number_of_courses_viewed          -0.023565   -0.004879   0.435914  
annual_income                      0.048618    0.005334   0.078256  
interaction_count                  1.000000    0.009888   0.374573  
lead_score                         0.009888    1.000000   0.193673  
converted                          0.374573    0.193673   1.000000  


In [9]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count")
]

for a, b in pairs:
    if a in data.columns and b in data.columns:
        corr = data[a].corr(data[b])
        print(f"Correlation between {a} and {b}: {corr:.3f}")

Correlation between interaction_count and lead_score: 0.010
Correlation between number_of_courses_viewed and lead_score: -0.005
Correlation between number_of_courses_viewed and interaction_count: -0.024
Correlation between annual_income and interaction_count: 0.049


In [11]:
from sklearn.model_selection import train_test_split

y = data['converted']
X = data.drop(columns=['converted'])  
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 877
Validation set size: 292
Test set size: 293


In [12]:
from sklearn.feature_selection import mutual_info_classif

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

mi_scores = {}
for col in categorical_cols:
    encoded = X_train[col].astype('category').cat.codes
    mi = mutual_info_classif(encoded.values.reshape(-1, 1), y_train, discrete_features=True)[0]
    mi_scores[col] = round(mi, 2)
mi_scores_sorted = dict(sorted(mi_scores.items(), key=lambda x: x[1], reverse=True))

mi_scores_sorted

{'lead_source': np.float64(0.03),
 'industry': np.float64(0.02),
 'employment_status': np.float64(0.02),
 'location': np.float64(0.0)}

In [13]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [18]:
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
numerical_cols = X_train.select_dtypes(include=[np.number]).columns

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[numerical_cols]), columns=numerical_cols)
X_val_num = pd.DataFrame(num_imputer.transform(X_temp[numerical_cols]), columns=numerical_cols)

X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[categorical_cols]), columns=categorical_cols)
X_val_cat = pd.DataFrame(cat_imputer.transform(X_temp[categorical_cols]), columns=categorical_cols)

encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded = pd.DataFrame(
    encoder.fit_transform(X_train_cat).toarray(),
    columns=encoder.get_feature_names_out(categorical_cols)
)
X_val_encoded = pd.DataFrame(
    encoder.transform(X_val_cat).toarray(),
    columns=encoder.get_feature_names_out(categorical_cols)
)

X_train_final = pd.concat([X_train_num.reset_index(drop=True), X_train_encoded.reset_index(drop=True)], axis=1)
X_val_final = pd.concat([X_val_num.reset_index(drop=True), X_val_encoded.reset_index(drop=True)], axis=1)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_final, y_train)

y_pred_val = model.predict(X_val_final)



In [19]:
accuracy = round(accuracy_score(y_temp, y_pred_val), 2)
accuracy

0.74

In [20]:
model_full = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_full.fit(X_train_final, y_train)
y_pred_full = model_full.predict(X_val_final)
base_accuracy = accuracy_score(y_temp, y_pred_full)

In [21]:
features_to_test = ['industry', 'employment_status', 'lead_score']


In [22]:
accuracy_diff = {}

for feature in features_to_test:
    feature_cols = [col for col in X_train_final.columns if feature in col]
    
    X_train_reduced = X_train_final.drop(columns=feature_cols, errors='ignore')
    X_val_reduced = X_val_final.drop(columns=feature_cols, errors='ignore')
    
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_reduced, y_train)
    y_pred = model.predict(X_val_reduced)
    acc = accuracy_score(y_temp, y_pred)    
    accuracy_diff[feature] = base_accuracy - acc



In [23]:
accuracy_diff

{'industry': 0.006837606837606924,
 'employment_status': 0.010256410256410331,
 'lead_score': 0.0034188034188034067}

In [24]:
C_values = [0.01, 0.1, 1, 10, 100]
accuracy_results = {}

for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train_final, y_train)
    y_pred_val = model.predict(X_val_final)
    acc = round(accuracy_score(y_temp, y_pred_val), 3)
    accuracy_results[C] = acc

accuracy_results

{0.01: 0.75, 0.1: 0.742, 1: 0.74, 10: 0.74, 100: 0.74}